In [2]:

import numpy as np
import pandas as pd

from scipy.special import expit
from scipy.stats import levy_stable

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

class MemoryBasedCuckooSearch:
    """
    Binary, memory-based Cuckoo Search for feature selection.

    Each nest is a binary vector:
        1 = select feature
        0 = exclude feature

    The fitness combines cross-validation error and
    the fraction of selected features.
    """

    def __init__(
        self,
        n_nests=10,
        n_iterations=100,
        pa=0.3,
        alpha=1.0,
        levy_lambda=1.5,
        feature_weight=0.1,
        memory_size=5,
        random_state=42,
    ):
        self.n_nests = n_nests
        self.n_iterations = n_iterations
        self.pa = pa
        self.alpha = alpha
        self.levy_lambda = levy_lambda
        self.feature_weight = feature_weight
        self.memory_size = memory_size
        self.random_state = random_state

    def _fitness(self, mask, X, y):
        """Lower fitness is better."""
        selected = np.flatnonzero(mask)

        # Every candidate must select at least one feature.
        if len(selected) == 0:
            return 1.0 + self.feature_weight

        # Keep the validation folds feasible for small datasets.
        _, class_counts = np.unique(y, return_counts=True)
        n_splits = min(5, int(class_counts.min()))

        if n_splits < 2:
            raise ValueError(
                "Each class needs at least two samples "
                "for stratified cross-validation."
            )

        model = make_pipeline(
            SimpleImputer(strategy="median"),
            StandardScaler(),
            SVC(kernel="linear", C=1.0),
        )

        cv = StratifiedKFold(
            n_splits=n_splits,
            shuffle=True,
            random_state=self.random_state,
        )

        scores = cross_val_score(
            model,
            X[:, selected],
            y,
            cv=cv,
            scoring="accuracy",
            n_jobs=-1,
        )

        error = 1.0 - scores.mean()
        feature_ratio = len(selected) / X.shape[1]

        return (
            (1.0 - self.feature_weight) * error
            + self.feature_weight * feature_ratio
        )

    def _levy_step(self, dimensions):
        """Generate a Lévy-distributed random step."""
        beta = self.levy_lambda

        # Mantegna-style Lévy-flight sampling.
        sigma_u = (
            (
                np.exp(
                    np.log(
                        np.math.gamma(1 + beta)
                    )
                )
                if False else 1.0
            )
        )

        # Stable distribution supplies heavy-tailed steps.
        step = levy_stable.rvs(
            beta, 0, size=dimensions, random_state=self.rng
        )
        return np.asarray(step, dtype=float)

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y)

        if X.ndim != 2:
            raise ValueError("X must be a 2D feature matrix.")

        self.rng = np.random.default_rng(self.random_state)
        n_samples, n_features = X.shape

        if n_features < 1:
            raise ValueError("At least one feature is required.")

        # Continuous positions are converted to binary masks
        # using a sigmoid probability.
        positions = self.rng.normal(
            0, 1, size=(self.n_nests, n_features)
        )
        nests = self.rng.random(
            (self.n_nests, n_features)
        ) < expit(positions)

        # Ensure that no nest is empty.
        for nest in nests:
            if not nest.any():
                nest[self.rng.integers(n_features)] = True

        scores = np.array([
            self._fitness(nest, X, y) for nest in nests
        ])

        best_idx = np.argmin(scores)
        best_mask = nests[best_idx].copy()
        best_score = scores[best_idx]

        # Memory holds historically improved feature subsets.
        memory = [(best_mask.copy(), float(best_score))]
        stagnation = np.zeros(self.n_nests, dtype=int)

        self.history_ = []

        for iteration in range(self.n_iterations):
            for i in range(self.n_nests):
                old_mask = nests[i].copy()

                # Lévy flight proposes a new continuous position.
                step = self._levy_step(n_features)
                position = positions[i] + self.alpha * step

                # Binary conversion via sigmoid probabilities.
                probability = expit(
                    np.clip(position, -20, 20)
                )
                candidate = self.rng.random(
                    n_features
                ) < probability

                if not candidate.any():
                    candidate[self.rng.integers(n_features)] = True

                candidate_score = self._fitness(candidate, X, y)

                # Greedy replacement if candidate is better.
                if candidate_score < scores[i]:
                    nests[i] = candidate
                    positions[i] = position
                    scores[i] = candidate_score
                    stagnation[i] = 0

                    memory.append(
                        (candidate.copy(), float(candidate_score))
                    )
                    memory.sort(key=lambda item: item[1])
                    memory = memory[:self.memory_size]
                else:
                    stagnation[i] += 1

                # Abandon nests probabilistically.
                if self.rng.random() < self.pa:
                    if (
                        stagnation[i] >= 3
                        and len(memory) > 0
                        and self.rng.random() < 0.5
                    ):
                        # Recover a remembered good solution and
                        # perturb it to continue exploring.
                        base = memory[
                            self.rng.integers(len(memory))
                        ][0].copy()

                        flips = self.rng.random(n_features) < 0.1
                        candidate = np.logical_xor(base, flips)
                    else:
                        candidate = (
                            self.rng.random(n_features) < 0.5
                        )

                    if not candidate.any():
                        candidate[self.rng.integers(n_features)] = True

                    candidate_score = self._fitness(candidate, X, y)

                    if candidate_score < scores[i]:
                        nests[i] = candidate
                        scores[i] = candidate_score
                        positions[i] = np.where(
                            candidate, 1.0, -1.0
                        )
                        stagnation[i] = 0

                        memory.append(
                            (candidate.copy(), float(candidate_score))
                        )
                        memory.sort(key=lambda item: item[1])
                        memory = memory[:self.memory_size]

                if scores[i] < best_score:
                    best_score = float(scores[i])
                    best_mask = nests[i].copy()

            self.history_.append({
                "iteration": iteration + 1,
                "best_fitness": float(best_score),
                "selected_features": int(best_mask.sum()),
            })

            if (
                iteration == 0
                or (iteration + 1) % 10 == 0
            ):
                print(
                    f"Iteration {iteration + 1:3d} | "
                    f"Fitness: {best_score:.4f} | "
                    f"Features: {best_mask.sum()}/{n_features}"
                )

        self.best_mask_ = best_mask
        self.best_fitness_ = best_score
        self.selected_features_ = np.flatnonzero(best_mask)

        return self

    def transform(self, X):
        X = np.asarray(X)
        return X[:, self.selected_features_]

    def fit_transform(self, X, y):
        return self.fit(X, y).transform(X)


# Load a built-in sample dataset
data = load_breast_cancer()
X = data.data
y = data.target

print("Original number of features:", X.shape[1])
print("Number of samples:", X.shape[0])

# Split the data before feature selection
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# Run Cuckoo Search feature selection
selector = MemoryBasedCuckooSearch(
    n_nests=10,
    n_iterations=20,  # Start small for testing
    pa=0.3,
    alpha=1.0,
    levy_lambda=1.5,
    feature_weight=0.1,
    memory_size=5,
    random_state=42
)

selector.fit(X_train, y_train)

X_train_selected = selector.transform(X_train)
X_test_selected = selector.transform(X_test)

# Train and evaluate the final SVM
model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    SVC(kernel="linear", C=1.0)
)

model.fit(X_train_selected, y_train)
predictions = model.predict(X_test_selected)

print("\nSelected features:", len(selector.selected_features_))
print("Test accuracy:", accuracy_score(y_test, predictions))
print(classification_report(y_test, predictions))

print("\nSelected feature names:")
print(data.feature_names[selector.selected_features_])

Original number of features: 30
Number of samples: 569
Iteration   1 | Fitness: 0.0644 | Features: 11/30
Iteration  10 | Fitness: 0.0525 | Features: 11/30
Iteration  20 | Fitness: 0.0471 | Features: 7/30

Selected features: 7
Test accuracy: 0.9473684210526315
              precision    recall  f1-score   support

           0       0.93      0.93      0.93        42
           1       0.96      0.96      0.96        72

    accuracy                           0.95       114
   macro avg       0.94      0.94      0.94       114
weighted avg       0.95      0.95      0.95       114


Selected feature names:
['mean perimeter' 'mean symmetry' 'concave points error' 'worst radius'
 'worst texture' 'worst perimeter' 'worst concave points']
